# California Housing: Regression and Regularization

**Goal:** Predict median house values and compare Linear Regression, Ridge, Lasso, and Elastic Net.

**Questions:** How well do the models predict unseen data? How does regularization affect error and coefficient size? Can Lasso/Elastic Net produce a simpler model?

**Data required:** `housing.csv` with `median_house_value` and `ocean_proximity`. The dataset is not bundled. Run all cells from top to bottom; results are generated by the notebook rather than hard-coded.

## 1. Imports and reproducibility

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED, TEST_SIZE, FOLDS = 42, 0.20, 5
CV = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", 100)

## 2. Load and inspect data

In [ ]:
DATA_PATH, TARGET = "housing.csv", "median_house_value"
df = pd.read_csv(DATA_PATH)
assert TARGET in df and "ocean_proximity" in df, "Check that the expected target and categorical columns exist."
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())
display(df.dtypes.rename("dtype").to_frame())
display(df.isna().sum().sort_values(ascending=False).rename("missing").to_frame())
display(df["ocean_proximity"].value_counts(dropna=False).to_frame("count"))
plt.hist(df[TARGET], bins=40)
plt.title("Distribution of median house value"); plt.xlabel("Median house value ($)"); plt.ylabel("Count"); plt.show()

## 3. Split data before fitting preprocessing

The held-out test set is reserved for final evaluation. Imputation, scaling, and encoding happen inside the pipeline, so each CV fold learns preprocessing only from that fold's training rows.

In [ ]:
X = df.drop(columns=TARGET).copy()
y = df[TARGET].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))])
preprocessor = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)],
                                 remainder="drop", verbose_feature_names_out=False)
print(f"Train: {len(X_train):,}; test: {len(X_test):,}")
print("Numeric:", num_cols)
print("Categorical:", cat_cols)

## 4. Regularization parameter conventions

- Ridge: `alpha = λ × n_train`, matching the objective `MSE + λ ||β||₂²`.
- Lasso: `alpha = λ/2`, matching `MSE + λ ||β||₁` under scikit-learn's half-MSE convention.
- Elastic Net: `alpha = λ1/2 + λ2`; `l1_ratio = (λ1/2)/alpha`.

For Lasso and Elastic Net, the target is divided by 100,000 during fitting so the penalty grid has a useful scale. Predictions and coefficients are converted back to dollars. Interpret those λ values under this scaled-target convention.

In [ ]:
class RidgeByLambda(BaseEstimator, RegressorMixin):
    def __init__(self, lambda_=0.01): self.lambda_ = lambda_
    def fit(self, X, y):
        self.model_ = LinearRegression() if self.lambda_ == 0 else Ridge(alpha=float(self.lambda_) * len(y))
        self.model_.fit(X, y); self.coef_ = np.ravel(self.model_.coef_); self.intercept_ = self.model_.intercept_
        return self
    def predict(self, X): return self.model_.predict(X)

class LassoByLambda(BaseEstimator, RegressorMixin):
    def __init__(self, lambda_=0.01, max_iter=100000): self.lambda_, self.max_iter = lambda_, max_iter
    def fit(self, X, y):
        self.model_ = Lasso(alpha=float(self.lambda_) / 2, max_iter=self.max_iter)
        self.model_.fit(X, np.asarray(y) / 1e5)
        self.coef_ = np.ravel(self.model_.coef_) * 1e5; self.intercept_ = self.model_.intercept_ * 1e5
        return self
    def predict(self, X): return self.model_.predict(X) * 1e5

class ElasticNetByLambdas(BaseEstimator, RegressorMixin):
    def __init__(self, lambda1=0.01, lambda2=0.01, max_iter=100000):
        self.lambda1, self.lambda2, self.max_iter = lambda1, lambda2, max_iter
    def fit(self, X, y):
        ys = np.asarray(y) / 1e5; l1, l2 = float(self.lambda1), float(self.lambda2)
        if l1 == 0 and l2 == 0: self.model_ = LinearRegression()
        elif l1 == 0: self.model_ = Ridge(alpha=l2 * len(ys))
        else:
            alpha = l1 / 2 + l2
            self.model_ = ElasticNet(alpha=alpha, l1_ratio=(l1 / 2) / alpha, max_iter=self.max_iter)
        self.model_.fit(X, ys); self.coef_ = np.ravel(self.model_.coef_) * 1e5
        self.intercept_ = self.model_.intercept_ * 1e5
        return self
    def predict(self, X): return self.model_.predict(X) * 1e5

def pipe(est): return Pipeline([("preprocess", preprocessor), ("regressor", est)])
def tune(est, grid):
    search = GridSearchCV(pipe(est), grid, scoring="neg_root_mean_squared_error", cv=CV,
                         n_jobs=-1, refit=True, return_train_score=True)
    search.fit(X_train, y_train)
    return search
print("Custom estimators ready.")

## 5. Cross-validation and hyperparameter selection

All tuning uses only the training split. Five-fold CV selects the regularization settings by mean RMSE; the test set is not used for selection.

In [ ]:
linear = pipe(LinearRegression())
linear_cv = -cross_val_score(linear, X_train, y_train, cv=CV,
                            scoring="neg_root_mean_squared_error", n_jobs=-1)
linear.fit(X_train, y_train)
ridge_search = tune(RidgeByLambda(), {"regressor__lambda_": [0, .01, .1, 1, 10, 100, 1000]})
lasso_search = tune(LassoByLambda(), {"regressor__lambda_": [.0001, .001, .01, .1, 1, 10]})
enet_search = tune(ElasticNetByLambdas(), {
    "regressor__lambda1": [0, .001, .01, .1, 1],
    "regressor__lambda2": [0, .001, .01, .1, 1]})

models = {"Linear Regression": linear, "Ridge": ridge_search.best_estimator_,
          "Lasso": lasso_search.best_estimator_, "Elastic Net": enet_search.best_estimator_}
cv_table = pd.DataFrame([
    {"Model":"Linear Regression", "CV RMSE":linear_cv.mean(), "Selected parameters":"No penalty"},
    {"Model":"Ridge", "CV RMSE":-ridge_search.best_score_, "Selected parameters":str(ridge_search.best_params_)},
    {"Model":"Lasso", "CV RMSE":-lasso_search.best_score_, "Selected parameters":str(lasso_search.best_params_)},
    {"Model":"Elastic Net", "CV RMSE":-enet_search.best_score_, "Selected parameters":str(enet_search.best_params_)}
]).sort_values("CV RMSE")
display(cv_table.style.format({"CV RMSE":"${:,.2f}"}))
print("Best Ridge:", ridge_search.best_params_)
print("Best Lasso:", lasso_search.best_params_)
print("Best Elastic Net:", enet_search.best_params_)
cv_table.sort_values("CV RMSE").plot(kind="bar", x="Model", y="CV RMSE", legend=False, rot=20,
                                     title="Mean cross-validated RMSE (training split)")
plt.ylabel("RMSE ($)"); plt.tight_layout(); plt.show()

### Cross-validation parameter sweeps

These plots show how cross-validated RMSE changes across the tested regularization strengths. They complement the selected-model comparison and help show whether stronger penalties improve or worsen performance.

In [ ]:
ridge_cv = pd.DataFrame(ridge_search.cv_results_).sort_values("param_regressor__lambda_")
lasso_cv = pd.DataFrame(lasso_search.cv_results_).sort_values("param_regressor__lambda_")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, table, title, param in [
    (axes[0], ridge_cv, "Ridge", "param_regressor__lambda_"),
    (axes[1], lasso_cv, "Lasso", "param_regressor__lambda_")]:
    x = table[param].astype(float).to_numpy()
    y_rmse = -table["mean_test_score"].to_numpy()
    mask = x > 0
    ax.plot(x[mask], y_rmse[mask], "o-")
    ax.set_xscale("log")
    ax.set_xlabel("lambda (log scale)")
    ax.set_ylabel("Mean CV RMSE ($)")
    ax.set_title(f"{title}: regularization vs CV error")
plt.tight_layout(); plt.show()

enet_cv = pd.DataFrame(enet_search.cv_results_)
enet_cv["lambda1"] = enet_cv["param_regressor__lambda1"].astype(float)
enet_cv["lambda2"] = enet_cv["param_regressor__lambda2"].astype(float)
enet_heatmap = enet_cv.pivot(index="lambda1", columns="lambda2", values="mean_test_score") * -1
sns.heatmap(enet_heatmap, annot=True, fmt=".0f")
plt.title("Elastic Net: mean CV RMSE ($)"); plt.xlabel("lambda2 (L2)"); plt.ylabel("lambda1 (L1)")
plt.tight_layout(); plt.show()

## 6. Final held-out test evaluation

In [ ]:
def coef_vector(model): return np.ravel(model.named_steps["regressor"].coef_)
def evaluate(name, model):
    pred_train, pred_test = model.predict(X_train), model.predict(X_test)
    b = coef_vector(model)
    return {"Model":name,
            "Train RMSE":np.sqrt(mean_squared_error(y_train, pred_train)),
            "Test RMSE":np.sqrt(mean_squared_error(y_test, pred_test)),
            "Test MAE":mean_absolute_error(y_test, pred_test),
            "Test R²":r2_score(y_test, pred_test),
            "Coefficient L2 norm":np.linalg.norm(b),
            "Coefficient L1 norm":np.linalg.norm(b, 1),
            "Non-zero coefficients":int(np.sum(np.abs(b) >= 1e-8)),
            "Total coefficients":len(b)}

test_table = pd.DataFrame([evaluate(k, v) for k, v in models.items()]).sort_values("Test RMSE")
display(test_table.style.format({"Train RMSE":"${:,.2f}", "Test RMSE":"${:,.2f}",
    "Test MAE":"${:,.2f}", "Test R²":"{:.4f}", "Coefficient L2 norm":"{:,.2f}",
    "Coefficient L1 norm":"{:,.2f}"}))

## 7. Coefficients and model complexity

Numeric features are standardized, so their coefficients represent approximate dollar changes per one-standard-deviation increase, holding other features fixed. Categorical coefficients are relative to the dropped reference category. Correlated features can make individual coefficients unstable; coefficients are not causal effects.

In [ ]:
fitted_prep = models["Linear Regression"].named_steps["preprocess"]
feature_names = fitted_prep.get_feature_names_out()
coef_table = pd.DataFrame({name: coef_vector(model) for name, model in models.items()}, index=feature_names)
display(coef_table.round(2))
coef_table.plot(kind="barh", figsize=(12, max(5, .45 * len(feature_names))))
plt.axvline(0, color="black", linewidth=.8)
plt.xlabel("Coefficient (dollars)"); plt.title("Coefficients for selected models")
plt.tight_layout(); plt.show()

complexity = test_table.set_index("Model")[["Coefficient L2 norm", "Coefficient L1 norm", "Non-zero coefficients"]]
display(complexity)
complexity[["Coefficient L2 norm", "Coefficient L1 norm"]].plot(kind="bar", title="Coefficient norms")
plt.ylabel("Norm (dollars)"); plt.tight_layout(); plt.show()

## 8. Prediction diagnostics

In [ ]:
winner = test_table.iloc[0]["Model"]
best = models[winner]
pred = best.predict(X_test)
residual = y_test.to_numpy() - pred
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(y_test, pred, alpha=.25, s=12)
lo, hi = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
ax[0].plot([lo, hi], [lo, hi], "--")
ax[0].set(xlabel="Actual value ($)", ylabel="Predicted value ($)", title=f"Actual vs predicted: {winner}")
ax[1].scatter(pred, residual, alpha=.25, s=12); ax[1].axhline(0, linestyle="--")
ax[1].set(xlabel="Predicted value ($)", ylabel="Residual ($)", title="Residuals vs predicted")
plt.tight_layout(); plt.show()

## 9. Evidence-based findings

This cell generates values from the run. Use them to write a short discussion; do not copy example metrics from an earlier run.

In [ ]:
cv_winner = cv_table.sort_values("CV RMSE").iloc[0]
test_winner = test_table.iloc[0]
print(f"Best CV model: {cv_winner['Model']} (mean CV RMSE ${cv_winner['CV RMSE']:,.2f}).")
print(f"Lowest test RMSE: {test_winner['Model']} (${test_winner['Test RMSE']:,.2f}); test R² = {test_winner['Test R²']:.4f}.")
print("\nNon-zero coefficients:")
for _, r in test_table.iterrows():
    print(f"{r['Model']}: {int(r['Non-zero coefficients'])}/{int(r['Total coefficients'])}")
print("\nDiscuss: (1) the practical size of RMSE differences in dollars; (2) whether regularization reduces complexity; "
      "(3) any difference between the CV winner and test winner; (4) limitations of linear models and capped target values.")

## 10. Conclusion template and limitations

Complete this after running all cells:

> The best model by cross-validation was **[model]**, with mean CV RMSE **[value]**. Its held-out test RMSE was **[value]** and test R² was **[value]**. Compared with ordinary linear regression, regularization **[improved / did not materially improve]** prediction. Coefficient analysis showed **[shrinkage / feature selection / both]**. Overall, **[evidence-based conclusion]**.

**Limitations:** the model is linear; correlated predictors complicate coefficient interpretation; the target values are capped; and results depend on the dataset version and train/test split.

**Before publishing:** run all cells from a clean kernel, remove personal identifiers and private notebook links, add a README and requirements file, and check course/data-sharing rules.